# BÁO CÁO ĐÁNH GIÁ DỮ LIỆU HUẤN LUYỆN VÀ TEST SET

# Hội thi Thử thách Trí tuệ Nhân tạo Thành phố Hồ Chí Minh 2026

## Hướng dẫn truy vấn dữ liệu thị giác dùng fiftyone

Đây là hướng dẫn dùng cho các đội tham dự AI Challenge 2026. Hướng dẫn này nhằm mục đích giới thiệu cho các đội một phương pháp cơ bản để truy vấn dữ liệu dựa trên thông tin BTC cung cấp và giới thiệu công cụ fiftyone để hỗ trợ đội thi đánh giá kết quả.

## Cài đặt ban đầu

Bạn cần cài đặt môi trường để chạy được notebook này trên máy tính cá nhân của bạn. Hướng dẫn này không bao gồm phần cài đặt môi trường. Khuyến nghị: các bạn có thể cài đặt [Anaconda](https://docs.anaconda.com/free/anaconda/install/windows/).

## Cài đặt các thư viện FiftyOne và PyTorch
Hướng dẫn này dùng fiftyone là công cụ để trực quan dữ liệu và pytorch là backend chính cho các thuật toán máy học.


In [1]:
! pip install fiftyone
! pip install torch torchvision torchaudio


zsh:1: command not found: pip
zsh:1: command not found: pip


Load dữ liệu keyframe từ thư mục chứa keyframe. Mỗi ảnh và thông tin đi kèm sau này sẽ được lưu trữ trong một Sample. Tất cả các Sample được lưu trong Dataset.

In [2]:
import fiftyone as fo
import fiftyone.brain as fob
import numpy as np
from glob import glob
import json
import os



/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load dữ liệu keyframe từ thư mục chứa keyframe. Trong hướng dẫn này tất cả các file Keyframes_L*.zip được giải nén vào thư mục `/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/keyframes`. Mỗi ảnh và thông tin đi kèm sau này sẽ được lưu trữ trong một `Sample`. Tất cả các `Sample` được lưu trong `Dataset`.

In [3]:
dataset = fo.Dataset.from_images_dir('/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/keyframes/keyframes', name=None, tags=None, recursive=True)

 100% |███████████| 110040/110040 [5.2s elapsed, 0s remaining, 21.4K samples/s]      


Sau khi dữ liệu đã load lên xong. Bạn có thể truy cập vào đường vào ứng dụng web của fiftyone từ [http://localhost:5151](http://localhost:5151)

In [4]:
session = fo.launch_app(dataset, auto=False)

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
Session launched. Run `session.show()` to open the App in a cell output.


Hoặc bạn có thể chạy cell bên dưới để mở tab mới cho ứng dụng web fiftyone

In [5]:
session.open_tab()

<IPython.core.display.Javascript object>

### Trích xuất thêm thông tin tên của video và frameid
Thông tin `video` và `frameid` sẽ được lấy từ tên của tập tin keyframe.

In [6]:
for sample in dataset:
    _, sample['video'], sample['frameid'] = sample['filepath'][:-4].rsplit('/', 2)
    sample.save()

Bạn có thể xem `Sample` đầu tiên của `Dataset` bằng lệnh sau:

In [7]:
print(dataset.first())

<Sample: {
    'id': '6a8863d9c62fa602d9e04f79',
    'media_type': 'image',
    'filepath': '/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/keyframes/keyframes/L21_V001/001.jpg',
    'tags': [],
    'metadata': None,
    'created_at': datetime.datetime(2026, 8, 21, 14, 42, 33, 421000),
    'last_modified_at': datetime.datetime(2026, 8, 21, 14, 42, 38, 612000),
    'video': 'L21_V001',
    'frameid': '001',
}>


### Thêm thông tin kết quả của object detection.

Bước này có thể tốn của bạn nhiều thời gian để đọc hết tất cả các dữ liệu về object detection. Bạn có thể bỏ qua cell này và chạy cell này sau nếu muốn thử thêm các thông tin về vector CLIP embedding trước.

In [8]:
for sample in dataset:
    object_path = f"/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/objects/objects/{sample['video']}/{sample['frameid']}.json"
    with open(object_path) as jsonfile:
        det_data = json.load(jsonfile)
    detections = []
    for cls, box, score in zip(det_data['detection_class_entities'], det_data['detection_boxes'], det_data['detection_scores']):
        # Convert to [top-left-x, top-left-y, width, height]
        boxf = [float(box[1]), float(box[0]), float(box[3]) - float(box[1]), float(box[2]) - float(box[0])]
        scoref = float(score)

        # Only add objects with confidence > 0.4
        if scoref > 0.4:
            detections.append(
                fo.Detection(
                    label=cls,
                    bounding_box= boxf,
                    confidence=float(score)
                )
            )
    sample["object_faster_rcnn"] = fo.Detections(detections=detections)
    sample.save()

### Thêm thông tin CLIP embedding.

In [9]:
import os

all_keyframe = glob('/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/keyframes/keyframes/**/*.jpg', recursive=True)
video_keyframe_dict = {}
all_video = sorted(set(os.path.basename(os.path.dirname(kf)) for kf in all_keyframe))

Đọc thông tin clip embedding được cung cấp.

Lưu ý: Các bạn cần tải đúng bản CLIP embedding từ model **CLIP ViT-B/32**

Tạo dictionary `video_keyframe_dict` với `video_keyframe_dict[video]` thông tin danh sách `keyframe` của `video`

In [10]:
for kf in all_keyframe:
    _, vid, kf = kf[:-4].rsplit('/',2)
    if vid not in video_keyframe_dict.keys():
        video_keyframe_dict[vid] = [kf]
    else:
        video_keyframe_dict[vid].append(kf)

Do thông tin vector CLIP embedding được cung cấp được lưu theo từng video nhầm mục đích tối ưu thời gian đọc dữ liệu. Cần sort lại danh sách `keyframe` của từng `video` để đảm bảo thứ tự đọc đúng với vector embedding được cung cấp.

In [11]:
for k,v in video_keyframe_dict.items():
    video_keyframe_dict[k] = sorted(v)

Tạo dictionary `embedding_dict` với `embedding_dict[video][keyframe]` lưu thông tin vector CLIP embedding của `keyframe` trong `video` tương ứng

In [12]:
embedding_dict = {}
for v in all_video:
    clip_path = f'/Users/dangphuong/AI/AIO2026/HCM/aic2026-Luminary/data_exam1/raw/clip-features/clip-features-32/{v}.npy'
    a = np.load(clip_path)
    embedding_dict[v] = {}
    for i,k in enumerate(video_keyframe_dict[v]):
        embedding_dict[v][k] = a[i]

Tạo danh sách `clip_embedding` ứng với danh sách `sample` trong `dataset`.

In [13]:
clip_embeddings = []
for sample in dataset:
    clip_embedding = embedding_dict[sample['video']][sample['frameid']]
    clip_embeddings.append(clip_embedding)


In [14]:
embedding_dict.keys()

dict_keys(['L21_V001', 'L21_V002', 'L21_V003', 'L21_V005', 'L21_V006', 'L21_V007', 'L21_V008', 'L21_V009', 'L21_V010', 'L21_V011', 'L21_V012', 'L21_V013', 'L21_V014', 'L21_V015', 'L21_V016', 'L21_V017', 'L21_V018', 'L21_V019', 'L21_V021', 'L21_V022', 'L21_V023', 'L21_V024', 'L21_V025', 'L21_V026', 'L21_V027', 'L21_V028', 'L21_V029', 'L21_V030', 'L21_V031', 'L22_V001', 'L22_V002', 'L22_V003', 'L22_V004', 'L22_V005', 'L22_V006', 'L22_V007', 'L22_V008', 'L22_V009', 'L22_V010', 'L22_V011', 'L22_V012', 'L22_V013', 'L22_V014', 'L22_V015', 'L22_V016', 'L22_V017', 'L22_V018', 'L22_V019', 'L22_V020', 'L22_V021', 'L22_V022', 'L22_V023', 'L22_V024', 'L22_V025', 'L22_V026', 'L22_V027', 'L22_V028', 'L22_V029', 'L22_V030', 'L22_V031', 'L23_V001', 'L23_V002', 'L23_V003', 'L23_V004', 'L23_V005', 'L23_V006', 'L23_V007', 'L23_V008', 'L23_V009', 'L23_V010', 'L23_V011', 'L23_V012', 'L23_V013', 'L23_V014', 'L23_V015', 'L23_V016', 'L23_V017', 'L23_V018', 'L23_V019', 'L23_V020', 'L23_V021', 'L23_V022', 'L23_

In [15]:
fob.compute_similarity(
    dataset,
    model="clip-vit-base32-torch",      # store model's name for future use
    embeddings=clip_embeddings,          # precomputed image embeddings
    brain_key="img_sim",
)

## Từ đây các bạn có thể thử các tính năng search, filter trên ứng dụng fiftyone.

In [16]:
# Bạn cần phải cài version umap-learn hỗ trợ.
# fob.compute_visualization(
#     dataset,
#     embeddings=clip_embeddings,
#     brain_key="img_viz"
# )
